# Plotly Express: 분석 질문에 맞는 시각화

이 노트북 하나를 Colab에 업로드하고 **N00의 설치 셀을 한 번 실행한 뒤, 원하는 실습 코드 셀을 선택해 실행**하세요. 별도 데이터 파일은 필요하지 않습니다. PPT의 N번호와 같은 절을 찾으면 됩니다. N번호는 실행 횟수 `In [ ]`와 다릅니다.

Plotly Express는 표의 열 이름을 받아 그래프와 범례를 구성하는 Plotly의 고수준 인터페이스입니다. 호버(점 위에 마우스 올리기), 확대, 범례 선택으로 관심 대상을 자세히 살펴볼 수 있습니다. `go`와 마찬가지로 결과는 `Figure`입니다.

이번 실습의 목표는 함수를 외우는 것이 아니라 **알아볼 질문에 맞춰 비교 대상을 선택하고, 필요한 값을 집계한 뒤, 적합한 그래프를 고르는 것**입니다. 각 절에서 질문과 자료를 먼저 읽고, 코드 실행 후 결과 해석을 확인하세요.

- 국가·식당·붓꽃 자료는 `px.data`에서 불러옵니다.
- 행사비·폴더·센서·지역·학습·시간별 이용 자료는 설명을 위해 코드로 만든 가상 자료입니다.


## N00 · 실행 준비

아래 설치 셀을 한 번 실행한 뒤, 라이브러리를 불러옵니다. 설치에는 잠시 시간이 걸릴 수 있습니다.

In [1]:
# N00
%pip -q install "plotly==7.1.0" "statsmodels>=0.14,<0.15"

You should consider upgrading via the '/Users/taeyoungkim/workspace/data_analysis_lecture/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# N00
import sys
import pandas as pd
import numpy as np
import plotly
import plotly.express as px
import plotly.io as pio

#pio.renderers.default = 'colab' if 'google.colab' in sys.modules else 'plotly_mimetype' # Colab의 경우 주석 해제
pio.templates.default = 'plotly_white'
gap = px.data.gapminder()
g07 = gap.loc[gap['year'] == 2007].copy()
print('Plotly 버전:', plotly.__version__)
print('Gapminder 자료:', gap.shape)


Plotly 버전: 7.1.0
Gapminder 자료: (1704, 8)


## N01 · Gapminder: 국가별 인구와 생활 수준

Gapminder에는 여러 나라의 인구, 기대수명, 1인당 GDP를 연도별로 정리한 자료가 들어 있습니다.

**자료 준비와 그래프 선택**

국가 간 비교는 같은 연도끼리, 한 국가의 변화는 여러 연도로 비교합니다. 자료의 기간·중복·결측부터 확인해야 비교 대상을 잘못 세지 않습니다.

**코드에서 확인할 부분**

- unique()는 중복을 뺀 값들을 돌려줍니다. ["A", "A", "B"]라면 결과는 ["A", "B"]입니다.
- nunique()는 서로 다른 값이 몇 개인지 셉니다. 같은 예제의 결과는 2입니다.
- 국가가 몇 개인지는 nunique(), 어떤 연도가 있는지는 unique()로 알아봅니다.

`country`는 국가, `continent`는 대륙, `year`는 관측 연도입니다. `lifeExp`는 기대수명(년), `pop`은 인구(명), `gdpPercap`은 1인당 GDP입니다. 기대수명은 개인의 실제 사망 나이가 아니며, 1인당 GDP는 개인 연봉이 아닙니다. 1952~2007년 자료이므로 현재의 국가 상황을 설명하는 데 그대로 쓰지 않습니다.

In [3]:
# N01
import plotly.express as px

gap = px.data.gapminder()
print(gap.head(), "\n")
print('행과 열:', gap.shape, "\n")
print('국가 수:', gap['country'].nunique(), "\n")
print('연도:', sorted(gap['year'].unique()), "\n")
print('결측값:', gap.isna().sum())

       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4   

행과 열: (1704, 8) 

국가 수: 142 

연도: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)] 

결측값: country      0
continent    0
year         0
lifeExp      0
pop          0
gdpPercap    0
iso_alpha    0
iso_num      0
dtype: int64


**이 결과에서 알 수 있는 것**

- 142개국을 12개 연도에 걸쳐 기록한 1,704행입니다. 한 국가가 여러 번 나옵니다.
- 2007년만 선택하면 142행이며, 국가별 비교에 쓸 수 있습니다.
- 축 눈금의 k는 1천, M은 100만, B는 10억입니다. 1B명은 10억 명입니다.

### 2007년만 남기면 행의 의미가 어떻게 달라질까?

In [4]:
# N01
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

g07 = gap.loc[gap['year'] == 2007].copy()
print(g07[['country', 'year', 'lifeExp', 'pop']].head())
print('2007년 기록 수:', len(g07))

        country  year  lifeExp       pop
11  Afghanistan  2007   43.828  31889923
23      Albania  2007   76.423   3600523
35      Algeria  2007   72.301  33333216
47       Angola  2007   42.731  12420476
59    Argentina  2007   75.320  40301927
2007년 기록 수: 142


**결과를 읽으면**

2007년 자료는 국가당 한 행입니다. 전체 자료를 한 시점의 자료처럼 해석하면 같은 국가를 반복해서 세게 됩니다.

현재 국가 상황을 판단하는 자료가 아닙니다. 먼저 비교할 시점과 대상을 정하세요.

## N02 · 산점도: 국가별 GDP와 기대수명

경제 수준이 높은 나라는 기대수명도 높을까요? 2007년의 국가별 자료를 산점도로 비교합니다.

**자료 준비와 그래프 선택**

2007년 자료를 선택하고 GDP와 기대수명 두 수치형 열을 사용합니다. 산점도는 값 두 개의 관계와 전체 경향에서 벗어난 대상을 함께 보여 줍니다.

**코드에서 확인할 부분**

- df에는 2007년 자료만 담습니다. 국가별로 같은 시점의 값을 비교하기 위해서입니다.
- x="gdpPercap"은 가로축에 1인당 GDP, y="lifeExp"는 세로축에 기대수명을 표시합니다.
- px.scatter()가 만든 Figure를 fig에 저장하고 fig.show()로 표시합니다.

In [5]:
# N02
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

df = gap.loc[gap['year'] == 2007].copy()
fig = px.scatter(df, x='gdpPercap', y='lifeExp')
fig.show()

**이 결과에서 알 수 있는 것**

- 국가 하나가 점 하나입니다. 오른쪽은 높은 GDP, 위쪽은 긴 기대수명입니다.
- GDP가 높은 국가에서 기대수명이 긴 경향이 있지만, 점들이 한 선 위에 놓이지는 않습니다.
- 왼쪽에 점이 몰리면 로그 축으로 다시 봅니다. 로그 축의 같은 간격은 같은 배수입니다.

### 가로축을 로그 눈금으로 바꾸면 어떤 점들이 잘 보일까?

In [ ]:
# N02
import plotly.express as px

# Gapminder에서 2007년 자료만 선택합니다.
gap = px.data.gapminder()
g07 = gap.loc[gap['year'] == 2007].copy()

fig = px.scatter(g07, x='gdpPercap', y='lifeExp', log_x=True)
fig.show()
# 왜 배수해서 확대하지 않고 log를 사용할까?

**결과를 읽으면**

로그 축에서는 같은 거리의 이동이 같은 차이가 아니라 같은 배수를 뜻합니다. 작은 GDP 값들이 겹쳐 있던 부분을 펼쳐 볼 수 있습니다.

이 그림으로 관계를 탐색할 수 있지만 GDP 증가가 수명 증가의 원인이라고 단정할 수는 없습니다.

## N03 · 호버: 점에 해당하는 국가 확인하기

산점도에서 눈에 띄는 점에 마우스를 올리면 국가 이름과 정확한 수치를 볼 수 있습니다.

**자료 준비와 그래프 선택**

같은 산점도에 국가 이름을 연결하고, 인구는 설명창에서 확인합니다. 호버는 점이 많은 그림을 글자로 채우지 않고 관심 대상의 상세값을 찾게 합니다.

**코드에서 확인할 부분**

- hover_name="country"는 마우스를 올린 점의 국가 이름을 설명창 맨 위에 표시합니다.
- hover_data=["pop"]을 추가하면 설명창에서 인구도 볼 수 있습니다.
- 추가 정보는 설명창에 나타나며, 점의 위치와 크기는 그대로 유지됩니다.

In [7]:
# N03
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

df = gap.loc[gap['year'] == 2007].copy()
fig = px.scatter(df, x='gdpPercap', y='lifeExp',
                 hover_name='country')
fig.show()

**이 결과에서 알 수 있는 것**

- 높거나 떨어져 있는 점에 마우스를 올려 어느 국가인지 확인합니다.
- hover_name은 국가 이름, hover_data는 함께 확인할 추가 열입니다.
- 관계 탐색은 전체 점으로, 정확한 수치 확인은 설명창으로 합니다.

### 호버에 출력 데이터 추가하기

In [8]:
# N03
import plotly.express as px

# Gapminder에서 2007년 자료만 선택합니다.
gap = px.data.gapminder()
g07 = gap.loc[gap['year'] == 2007].copy()

fig = px.scatter(g07, x='gdpPercap', y='lifeExp',
                 hover_name='country', hover_data=['pop', 'continent'])
fig.show()

**결과를 읽으면**

인구와 대륙은 설명창에만 추가됩니다. 시각적으로 표현하는 변수와 상세 확인용 변수는 다를 수 있습니다.

PPT는 정지 그림입니다. Colab에서 관심 국가 두 곳의 이름과 수치를 확인하세요.

## N04 · LOWESS: 구간에 따라 달라지는 추세

GDP가 늘어날 때 기대수명은 같은 속도로 높아질까요? 곡선으로 구간별 경향을 살펴봅니다.

**자료 준비와 그래프 선택**

같은 연도의 GDP·기대수명 산점도에 LOWESS 추세 곡선을 추가합니다. 증가 폭이 구간마다 다를 때는 하나의 직선보다 곡선이 경향을 잘 드러냅니다.

**코드에서 확인할 부분**

- LOWESS(국소 가중 평활)는 가까운 점에 더 큰 가중치를 주어 추세 곡선을 구합니다.
- trendline="ols"는 직선, "lowess"는 구간에 따라 기울기가 달라지는 곡선을 그립니다.
- frac은 계산에 참고할 주변 점의 비율입니다. 사용하려면 statsmodels가 필요합니다.

In [9]:
# N04
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

df = gap.loc[gap['year'] == 2007].copy()
fig = px.scatter(df, x='gdpPercap', y='lifeExp',
                 hover_name='country', trendline='lowess')
fig.show()

**이 결과에서 알 수 있는 것**

- 낮은 GDP 구간에서는 가파르고, 높은 구간에서는 상대적으로 완만한 경향이 보입니다.
- LOWESS는 가까운 점을 참고해 구간별 경향을 구합니다. 미래 예측선은 아닙니다.
- frac을 작게 하면 세부 굴곡에 민감해집니다. 설정을 바꿔도 남는 경향인지 비교합니다.

### 주변 자료를 넓게 또는 좁게 참고하면 선은 어떻게 달라질까?

In [10]:
# N04
import plotly.express as px

# Gapminder에서 2007년 자료만 선택합니다.
gap = px.data.gapminder()
g07 = gap.loc[gap['year'] == 2007].copy()

for frac in [0.3, 0.8]:
    fig = px.scatter(g07, x='gdpPercap', y='lifeExp',
                     trendline='lowess', trendline_options={'frac': frac},
                     title=f'LOWESS: frac={frac}')
    fig.show()

**결과를 읽으면**

frac은 각 국소 추정에서 참고하는 자료의 비율입니다. 작으면 굴곡에 민감하고, 크면 전반적인 경향을 더 부드럽게 보여 줍니다.

점이 드문 구간의 작은 굴곡을 중요한 현상으로 해석하지 않습니다.

## N05 · 색으로 대륙을 구분하기

국가별 점에 대륙마다 다른 색을 주면 어떤 대륙의 국가들이 비슷한 위치에 모이는지 알 수 있습니다.

**자료 준비와 그래프 선택**

위치에는 GDP·기대수명, 색에는 대륙 이름을 지정합니다. 집단을 색으로 구분하면 전체 관계가 집단 안에서도 비슷한지 살펴볼 수 있습니다.

**코드에서 확인할 부분**

- color="continent"는 같은 대륙의 국가에 같은 색을 지정합니다.
- 범례(legend)는 각 색이 어느 대륙을 나타내는지 알려 줍니다.
- color="lifeExp"로 바꾸면 기대수명에 따라 색이 달라지고, 색상 막대에 수치가 표시됩니다.

In [11]:
# N05
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

df = gap.loc[gap['year'] == 2007].copy()
fig = px.scatter(df, x='gdpPercap', y='lifeExp',
                 hover_name='country', color='continent')
fig.show()

**이 결과에서 알 수 있는 것**

- 같은 대륙의 점들이 모이는 위치와, 그 안에서 벗어난 국가를 찾습니다.
- 대륙 색에는 크기 순서가 없습니다. 기대수명을 색으로 지정하면 수치의 크기를 뜻합니다.
- 범례를 눌러 관심 대륙만 남기고, 전체를 볼 때 놓친 차이를 확인합니다.

### 대륙 대신 기대수명을 색에 지정하면 범례는 어떻게 바뀔까?

In [12]:
# N05
import plotly.express as px

# Gapminder에서 2007년 자료만 선택합니다.
gap = px.data.gapminder()
g07 = gap.loc[gap['year'] == 2007].copy()

fig = px.scatter(g07, x='gdpPercap', y='lifeExp',
                 color='lifeExp', hover_name='country',
                 color_continuous_scale='Viridis')
fig.show()

**결과를 읽으면**

색이 집단 이름 대신 기대수명이라는 수치의 크기를 표현합니다. 연속 색상 막대에서 색과 수치의 대응을 읽어야 합니다.

색에 집단 이름을 넣는지 수치를 넣는지에 따라 범례와 해석이 달라집니다.

## N06 · 버블 차트: 인구를 원의 크기로 표현하기

GDP와 기대수명을 비교하면서 인구 규모도 함께 보려면, 인구가 많은 국가를 더 큰 원으로 표시합니다.

**자료 준비와 그래프 선택**

같은 2007년 자료에서 인구를 원의 면적에 추가로 표현합니다. 국가별 관계와 함께, 많은 인구가 어떤 경제·건강 수준에 놓이는지 봅니다.

**코드에서 확인할 부분**

- size="pop"은 인구에 비례하도록 원의 면적을 정합니다.
- size_max는 가장 큰 원의 표시 크기를 조절합니다. 국가별 인구 비율은 바뀌지 않습니다.
- x와 y는 원의 위치, color는 대륙, size는 인구를 나타냅니다.

In [13]:
# N06
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

df = gap.loc[gap['year'] == 2007].copy()
fig = px.scatter(df, x='gdpPercap', y='lifeExp',
                 hover_name='country', color='continent',
                 size='pop', size_max=60)
fig.show()

**이 결과에서 알 수 있는 것**

- 원의 위치는 GDP·기대수명, 면적은 인구, 색은 대륙입니다.
- 큰 원이 왼쪽에 있으면 인구는 많지만 1인당 GDP는 상대적으로 낮은 국가입니다.
- size_max를 줄이면 겹침이 줄어듭니다. 정확한 인구는 호버로 확인합니다.

## N07 · 패싯: 대륙별로 나누어 보는 산점도

모든 국가를 한 그림에 놓으면 점이 겹칩니다. 대륙별로 그림을 나누면 각 대륙의 특징이 잘 보입니다.

**자료 준비와 그래프 선택**

대륙별로 그림을 나누되 x·y축의 범위와 눈금은 같게 유지합니다. 색만으로 구분하기 어려울 때 집단별 분포와 예외를 나누어 비교합니다.

**코드에서 확인할 부분**

- 패싯(facet)은 같은 형식의 그래프를 집단별로 나누어 배치하는 방법입니다.
- facet_col="continent"는 대륙별 산점도를 가로로 나란히 놓습니다. col은 column의 약자입니다.
- 모든 그림에 같은 축 범위를 사용하므로 점의 높이와 가로 위치를 서로 비교할 수 있습니다.

In [14]:
# N07
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

df = gap.loc[gap['year'] == 2007].copy()
fig = px.scatter(df, x='gdpPercap', y='lifeExp',
                 hover_name='country', color='continent',
                 size='pop', size_max=60,
                 facet_col='continent')
fig.show()

**이 결과에서 알 수 있는 것**

- 패싯(facet)은 같은 그래프를 집단별로 나눈 배치입니다. 새 통계량은 아닙니다.
- 유럽은 높은 기대수명 구간에, 아프리카는 더 넓은 기대수명 범위에 놓입니다.
- 축 범위가 같아야 위치를 직접 비교할 수 있습니다. 국가 수가 적은 집단도 확인합니다.

### 대륙 다섯 개를 한 줄에 놓기 어렵다면?

In [15]:
# N07
import plotly.express as px

# Gapminder에서 2007년 자료만 선택합니다.
gap = px.data.gapminder()
g07 = gap.loc[gap['year'] == 2007].copy()

fig = px.scatter(g07, x='gdpPercap', y='lifeExp',
                 color='continent', facet_col='continent',
                 facet_col_wrap=3, hover_name='country')
fig.show()

**결과를 읽으면**

facet_col_wrap=3은 한 줄에 최대 세 칸을 놓습니다. 비교 기준은 유지하면서 각 칸의 가로 공간을 늘릴 수 있습니다.

Colab에서 한 줄의 칸 수를 바꾸어 보세요. 배치가 바뀌어도 데이터와 비교 기준은 같습니다.

## N08 · 애니메이션: 국가별 변화 따라가기

연도를 바꾸며 국가별 점의 이동을 보면 GDP와 기대수명이 어떻게 변했는지 알 수 있습니다.

**자료 준비와 그래프 선택**

여러 연도를 유지하고 국가 이름으로 같은 대상을 추적합니다. 축은 고정합니다. 애니메이션은 개별 국가의 이동을 보여 줍니다. 연도별 정확한 비교는 선그래프도 유용합니다.

**코드에서 확인할 부분**

- animation_frame="year"는 연도별 그림을 만들어 순서대로 보여 줍니다.
- animation_group="country"는 연도가 바뀌어도 같은 국가의 점을 이어 줍니다.
- range_x와 range_y로 축 범위를 고정해, 모든 연도를 같은 눈금으로 비교합니다.

In [16]:
# N08
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

df = gap.copy()
fig = px.scatter(df, x='gdpPercap', y='lifeExp',
    hover_name='country', color='continent', size='pop', size_max=60,
    animation_frame='year', animation_group='country',
    range_x=[100, 120000], range_y=[20, 90])
fig.show()

**이 결과에서 알 수 있는 것**

- 오른쪽 이동은 GDP 증가, 위쪽 이동은 기대수명 증가입니다. 원 크기는 인구입니다.
- 슬라이더의 연도는 5년 간격 관측입니다. 중간 모든 연도의 값을 측정한 것은 아닙니다.
- 한국·일본만 남기면 두 국가의 변화를 따라가기 쉽습니다. N08 추가 실험에서 비교합니다.

### 두 나라만 남겨 경로를 따라가면?

In [17]:
# N08
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

selected = gap.loc[gap['country'].isin(['Korea, Rep.', 'Japan'])]
fig = px.scatter(selected, x='gdpPercap', y='lifeExp',
    color='country', size='pop', animation_frame='year',
    animation_group='country', range_x=[0, 40000], range_y=[20, 90])
fig.show()

**결과를 읽으면**

같은 국가를 추적하기가 쉬워집니다. 중간 연도 변화는 데이터에 있는 5년 간격 관측으로 확인하며, 사이의 모든 연도를 측정한 것은 아닙니다.

축이 장면마다 바뀌면 점의 이동을 잘못 읽을 수 있으므로 range_x·range_y를 고정합니다.

## N09 · 3차원 산점도: 인구를 세 번째 축에 표시하기

GDP, 기대수명, 인구를 각각 x, y, z축에 놓으면 국가별 세 가지 특성을 한 점으로 나타낼 수 있습니다.

**자료 준비와 그래프 선택**

GDP·기대수명·인구를 각각 x·y·z축에 놓고 대륙을 색으로 구분합니다. 세 수치를 한 번에 탐색할 수 있지만, 겹친 점은 시점을 바꿔 확인해야 합니다.

**코드에서 확인할 부분**

- px.scatter_3d()의 x, y, z에 각 축에 표시할 열 이름을 지정합니다.
- 인구는 z축의 좌표입니다. 인구가 많은 국가는 z값이 큰 위치에 놓입니다.
- 노트북에서 마우스로 그림을 회전하면 앞쪽 점에 가려진 국가도 찾아볼 수 있습니다.

In [18]:
# N09
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

df = gap.loc[gap['year'] == 2007].copy()
fig = px.scatter_3d(df, x='gdpPercap', y='lifeExp', z='pop',
                    hover_name='country', color='continent')
fig.update_traces(marker_size=1)
fig.show()

**이 결과에서 알 수 있는 것**

- 높은 z값은 큰 인구입니다. 화면 위쪽에 보인다는 사실만으로 인구를 판단하지 않습니다.
- Colab에서 회전하면 가려진 점이 드러납니다. 정지 화면 하나로 거리를 확정하지 않습니다.
- 두 변수의 정확한 비교가 목적이면 2차원 산점도나 패싯을 선택합니다.

### 꽃의 세 가지 길이로 품종을 구분할 수 있을까?

In [19]:
# N09
import plotly.express as px

iris = px.data.iris()
fig = px.scatter_3d(iris, x='sepal_length', y='petal_length',
                    z='petal_width', color='species')
fig.update_traces(marker_size=1)
fig.show()

**결과를 읽으면**

이번에는 점 하나가 국가가 아니라 꽃 한 송이입니다. 품종별로 모이는 영역을 탐색할 수 있지만 이 그림 자체가 분류 모델은 아닙니다.

차원이 많다고 항상 더 좋은 그래프는 아닙니다. 설명하려는 관계가 두 개라면 2차원이 더 쉽습니다.

## N10 · 누적 막대그래프: 대륙별 인구와 국가 구성

대륙별 인구를 비교하고, 각 대륙에서 어느 국가가 큰 비중을 차지하는지도 함께 살펴봅니다.

**자료 준비와 그래프 선택**

2007년 국가별 인구를 대륙별로 쌓아 합계와 국가 구성을 함께 봅니다. 총량과 세부 기여를 함께 보려면 누적 막대, 총량만 보려면 집계한 막대가 적합합니다.

**코드에서 확인할 부분**

- 국가별 인구를 같은 대륙의 막대에 차례로 쌓습니다. 막대 전체 길이는 인구 합계입니다.
- color="lifeExp"는 막대의 각 국가 조각에 기대수명에 따른 색을 지정합니다.
- 옆으로 나란히 놓는 그룹 막대와 달리, 누적 막대는 국가별 값을 이어 붙여 합계를 보여 줍니다.

In [20]:
# N10
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

df = gap.loc[gap['year'] == 2007].copy()
fig = px.bar(df, x='pop', y='continent',
             color='lifeExp', hover_name='country')
fig.show()

**이 결과에서 알 수 있는 것**

- 막대 전체 길이는 해당 대륙에 포함된 국가들의 인구 합계입니다.
- 조각 길이는 국가 인구, 조각 색은 기대수명입니다. 길이와 색은 서로 다른 값입니다.
- 대륙별 총인구만 궁금하면 groupby로 합산해 대륙당 막대 하나로 그립니다.

### 국가 조각을 없애고 대륙 합계만 비교한다면?

In [21]:
# N10
import plotly.express as px

# Gapminder에서 2007년 자료만 선택합니다.
gap = px.data.gapminder()
g07 = gap.loc[gap['year'] == 2007].copy()

total = g07.groupby('continent', as_index=False)['pop'].sum()
fig = px.bar(total, x='continent', y='pop', color='continent')
fig.show()
print(total)

  continent         pop
0    Africa   929539692
1  Americas   898871184
2      Asia  3811953827
3    Europe   586098529
4   Oceania    24549947


**결과를 읽으면**

합계를 먼저 계산하면 대륙당 하나의 막대가 됩니다. 원래 그래프의 총길이와 같은 양을 더 단순하게 표현합니다.

추가 실험에서 국가 조각을 없애도 대륙의 인구 합계는 변하지 않는지 확인합니다.

## N11 · 애니메이션: 대륙별 인구 변화

연도별 누적 막대를 재생하면 대륙의 인구 총량과 국가별 구성 변화를 함께 볼 수 있습니다.

**자료 준비와 그래프 선택**

국가별 인구를 대륙별로 쌓고 연도마다 갱신합니다. 높이·색 범위를 고정합니다. 대륙별 총인구와 국가 구성이 시간에 따라 바뀌는 모습을 함께 봅니다.

**코드에서 확인할 부분**

- animation_frame="year"는 막대그래프를 연도별로 바꿔 보여 줍니다.
- 막대 높이는 인구를, 조각의 색은 해당 국가의 기대수명을 나타냅니다.
- range_y와 range_color를 고정해 연도가 바뀌어도 높이와 색의 기준을 유지합니다.

In [22]:
# N11
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

fig = px.bar(gap, x='continent', y='pop', color='lifeExp',
    hover_name='country', animation_frame='year',
    animation_group='country', range_y=[0, 4000000000],
    range_color=[20, 85])
fig.show()

**이 결과에서 알 수 있는 것**

- 연도가 바뀔 때 막대 높이로 대륙 인구의 증감을 확인합니다. 색은 기대수명입니다.
- 국가 조각을 대륙 합계로 집계하면 추적 대상도 country에서 continent로 바뀝니다.
- 특정 두 연도의 차이를 정확히 설명하려면 해당 수치나 선그래프를 함께 확인합니다.

### 대륙 합계만 애니메이션으로 보여 준다면?

In [36]:
# N11
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

total = gap.groupby(['year', 'continent'], as_index=False)['pop'].sum()
fig = px.bar(total, x='continent', y='pop', color='continent',
             animation_frame='year', animation_group='continent',
             range_y=[0, 4000000000])
fig.show()

**결과를 읽으면**

장면 사이에 추적하는 대상도 country에서 continent로 바뀝니다. 집계 후 행의 단위에 맞춰 animation_group을 정합니다.

애니메이션은 변화 관찰에, 선그래프는 여러 시점을 동시에 비교하는 데 유리합니다.

## N12 · 선그래프: 한국의 기대수명 변화

한국의 기대수명이 시간에 따라 얼마나 높아졌는지 연도별 선그래프로 살펴봅니다.

**자료 준비와 그래프 선택**

한국만 선택하고 year로 정렬한 뒤 기대수명을 연도순으로 연결합니다. 시간 순서가 중요한 자료에서는 선의 방향과 기울기로 변화를 읽습니다.

**코드에서 확인할 부분**

- country가 "Korea, Rep."인 자료를 선택해 한국의 기록만 남깁니다.
- sort_values("year")로 연도를 정렬합니다. 선은 입력한 순서대로 점을 연결합니다.
- 가로축은 연도, 세로축은 기대수명입니다. 같은 기간에 선이 가파를수록 증가량이 큽니다.

In [23]:
# N12
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

df = gap.loc[gap['country'] == 'Korea, Rep.'].sort_values('year')
fig = px.line(df, x='year', y='lifeExp')
fig.show()

**이 결과에서 알 수 있는 것**

- 한국의 기대수명은 1952년 약 47.5년에서 2007년 약 78.6년으로 늘었습니다.
- 같은 기간에 더 가파른 구간은 기대수명 증가량이 더 큽니다.
- 일본과 함께 그리면 같은 연도의 세로 간격이 두 나라의 기대수명 차이입니다.

### 한국과 일본을 같은 축에서 비교한다면?

In [24]:
# N12
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

df = gap.loc[gap['country'].isin(['Korea, Rep.', 'Japan'])]
df = df.sort_values('year')
fig = px.line(df, x='year', y='lifeExp', color='country', markers=True)
fig.show()

## N13 · 선그래프: 국가별 인구 변화 비교

국가별 인구 변화를 한 그림에 그리면 인구 규모와 증가 추세를 비교할 수 있습니다.

**자료 준비와 그래프 선택**

인구 규모를 비교할지, 시작 시점 이후의 증가 배수를 비교할지 먼저 정합니다. 절대 인구는 규모를, 시작 연도 인구로 나눈 값은 상대적인 성장을 보여 줍니다.

**코드에서 확인할 부분**

- color="country"는 국가마다 선의 색을 다르게 지정하고 국가명을 범례에 표시합니다.
- 중국과 인도처럼 인구가 많은 국가 때문에 작은 국가의 변화는 아래쪽에 몰려 보입니다.
- 몇 배로 늘었는지 비교하려면 각 국가의 인구를 그 국가의 시작 연도 인구로 나눕니다.

In [ ]:
# N13
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
# 문제점 : 선이 너무 많다. 2개의 국가만 튀어나와서 다른 국가들 비교가 불가능하다.
gap = px.data.gapminder()

df = gap.sort_values('year')
fig = px.line(df, x='year', y='pop', color='country')
fig.show()

**이 결과에서 알 수 있는 것**

- 절대 인구를 그리면 큰 국가가 두드러지고 작은 국가의 변화는 아래쪽에 겹칩니다.
- 추가 실험은 각 국가의 인구를 그 국가의 1952년 인구로 나눕니다.
- 모두 1에서 시작하며 2는 두 배입니다. 큰 인구와 큰 증가 배수는 다른 기준입니다.

### 인구수를 시작 연도 대비 증가 배수로 비교한다면?

In [26]:
# N13
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

df = gap.loc[gap['country'].isin(['China', 'India', 'Korea, Rep.'])].copy()
#print(df.head(10))
# 국가별 1952년 인구를 대응표로 만듭니다.
base = df.loc[df['year'] == 1952].set_index('country')['pop']
#print(base.head())
# 각 국가의 인구를 그 국가의 시작 인구로 나눕니다.
df['배수'] = df['pop'] / df['country'].map(base)
fig = px.line(df.sort_values('year'), x='year', y='배수', color='country')
fig.show()

## N14 · 누적 영역그래프: 인구 총량과 구성 변화

대륙별 인구를 색칠한 영역으로 쌓으면 전체 인구와 대륙별 비중이 어떻게 변하는지 볼 수 있습니다.

**자료 준비와 그래프 선택**

연도를 정렬하고 국가별 인구를 쌓습니다. 같은 대륙에는 같은 색을 줍니다. 누적 영역은 시간에 따른 전체 규모와 부분별 기여를 함께 보여 줍니다.

**코드에서 확인할 부분**

- px.area()는 선 아래를 색칠하고 각 국가의 값을 위로 쌓습니다.
- line_group="country"는 같은 국가의 연도별 값을 하나의 선으로 연결합니다.
- color="continent"는 같은 대륙에 같은 색을 사용합니다. 대륙 인구는 색 영역의 두께로 읽습니다.

In [28]:
# N14
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

fig = px.area(gap.sort_values('year'), x='year', y='pop',
              color='continent', line_group='country')
fig.show()

**이 결과에서 알 수 있는 것**

- 맨 위 경계의 높이는 이 자료에 포함된 국가들의 인구 합계입니다.
- 대륙 인구는 색 영역의 위·아래 경계 사이 두께입니다. 위쪽 경계값 자체가 아닙니다.
- 대륙별로 먼저 합산하면 내부의 국가별 선이 줄어 전체 구성이 쉽게 보입니다.

### 대륙당 하나의 영역만 남긴다면?

In [29]:
# N14
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

total = gap.groupby(['year', 'continent'], as_index=False)['pop'].sum()
fig = px.area(total, x='year', y='pop', color='continent')
fig.show()

**결과를 읽으면**

총량은 유지하면서 대륙별 구성 변화가 선명해집니다. 한 대륙의 작은 변화는 독립된 선그래프가 더 정확히 보일 수 있습니다.

각 대륙의 작은 증감만 비교하려면 공통 기준선이 있는 별도 선그래프가 더 읽기 쉽습니다.

## N15 · 원그래프: 아메리카 국가별 인구 비율

2007년 아메리카 국가들의 인구를 합쳤을 때, 각 국가가 차지하는 비율을 비교합니다.

**자료 준비와 그래프 선택**

2007년 아메리카 국가만 선택합니다. 이 국가들의 인구 합계를 100%로 봅니다. 전체에서 차지하는 비중은 원그래프, 비슷한 값의 순위는 막대그래프로 비교합니다.

**코드에서 확인할 부분**

- year가 2007이고 continent가 Americas인 자료만 선택합니다.
- names는 국가 이름, values는 비율 계산에 사용할 인구입니다.
- 각 국가의 비율은 국가 인구를 선택한 아메리카 국가들의 인구 합계로 나누어 구합니다.

In [33]:
# N15
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
# 문제점 : 작은 부분의 비교가 어렵다.
gap = px.data.gapminder()

df = gap.loc[(gap['year'] == 2007) & (gap['continent'] == 'Americas')]
fig = px.pie(df, values='pop', names='country')
fig.update_traces(textposition='inside', textinfo='label+percent')
fig.update_layout(uniformtext_minsize=12,uniformtext_mode='hide')
fig.show()

**이 결과에서 알 수 있는 것**

- 한 조각은 국가 하나이며 중심각은 선택한 전체 인구에서 차지하는 비율입니다.
- 여기서 100%는 전 세계가 아니라 자료에 포함된 아메리카 국가들의 인구입니다.

### 막대 그래프로 순위를 비교하면?

In [ ]:
# N15
import plotly.express as px

# Gapminder에서 2007년 자료만 선택합니다.
# 막대 그래프는 작은 비교도 어느정도 가능하다.
gap = px.data.gapminder()
g07 = gap.loc[gap['year'] == 2007].copy()

df = g07.loc[g07['continent'] == 'Americas'].sort_values('pop')
fig = px.bar(df, x='pop', y='country', orientation='h', height=750)
fig.show()

**결과를 읽으면**

같은 자료를 막대 길이로 비교합니다. 구성 비율보다 국가 간 크기 비교가 목적일 때 유리합니다.

구성 비율을 설명할 때는 전체가 무엇인지 먼저 밝힙니다.

## N16 · 선버스트: 대륙 안의 국가별 인구 비율

대륙별 인구 비율과 각 대륙 안의 국가별 비율을 안쪽과 바깥쪽 고리로 함께 보여 줍니다.

**자료 준비와 그래프 선택**

대륙·국가의 계층을 만들고, 합산할 값에는 인구를 지정합니다. 선버스트는 전체 구성과 상위 항목 안의 세부 구성을 함께 보여 줍니다.

**코드에서 확인할 부분**

- path는 분류 순서입니다. 안쪽에 대륙을 놓고 그 바깥에 해당 대륙의 국가들을 놓습니다.
- values="pop"은 인구에 비례해 각 조각의 중심각을 정합니다.
- color="lifeExp"는 기대수명을 색으로 표시합니다. 대륙 색에는 인구로 가중한 평균을 사용합니다.

In [ ]:
# N16
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

df = gap.loc[gap['year'] == 2007].copy()
#path 순서가 안쪽부터다. 안쪽 원 == continent, 바깥 원 == country
#더 큰 범주(부모 node)가 안쪽으로 들어가야 깔끔하게 보여줄 수 있다
fig = px.sunburst(df, path=['continent', 'country'],
                  values='pop', color='lifeExp')
fig.show()

In [34]:
# N16
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

df = gap.loc[gap['year'] == 2007].copy()
fig = px.sunburst(df, path=[px.Constant('Earth'), 'continent', 'country'],
                  values='pop', color='lifeExp')
fig.show()

**이 결과에서 알 수 있는 것**

- 안쪽 대륙은 부모, 바깥 국가는 자식입니다. 조각의 중심각은 인구를 나타냅니다.
- 색은 기대수명입니다. 대륙의 색은 국가별 기대수명을 인구로 가중한 평균입니다.
- 국가마다 1을 넣어 합산하면 국가 수 비중으로 바뀝니다. 인구 비중과 다른 질문입니다.

## N27 · 동아리 행사비: 어디에 돈을 많이 썼을까?

공연부, 홍보부, 운영부의 행사비를 부서별·항목별로 정리한 가상 자료입니다.

**자료 준비와 그래프 선택**

가상 행사비를 부서·항목별로 나누고 지출 금액을 합산합니다. 돈이 많이 든 곳을 찾으려면 금액, 결제가 잦은 곳을 찾으려면 건수를 봅니다.

**코드에서 확인할 부분**

- path=["부서", "항목"]으로 안쪽에는 부서, 바깥쪽에는 세부 지출 항목을 놓습니다.
- values="금액"은 지출액이 큰 항목의 조각을 크게 표시합니다.
- values를 영수증 건수로 바꾸면 지출액 대신 결제 횟수의 비율을 비교할 수 있습니다.

### N27 · 실습에 사용할 가상 자료 만들기

다음 셀에서 표를 만듭니다. 파일을 불러올 필요는 없습니다. 표를 확인한 뒤 아래 셀에서 그래프를 그립니다.

In [ ]:
# N27
# pandas와 연계한 plotly
import pandas as pd

club_budget = pd.DataFrame({
    '부서': ['공연부', '공연부', '공연부', '홍보부', '홍보부', '운영부', '운영부'],
    '항목': ['무대', '음향', '소품', '인쇄', '광고', '식수', '안내'],
    '금액': [1500000, 800000, 200000, 150000, 450000, 180000, 120000],
    '건수': [1, 1, 12, 3, 2, 20, 10]
})
print(club_budget)

    부서  항목       금액  건수
0  공연부  무대  1500000   1
1  공연부  음향   800000   1
2  공연부  소품   200000  12
3  홍보부  인쇄   150000   3
4  홍보부  광고   450000   2
5  운영부  식수   180000  20
6  운영부  안내   120000  10


In [36]:
# N27
import pandas as pd
import plotly.express as px

# 이 예제에서 사용할 가상 자료를 새로 만듭니다.
club_budget = pd.DataFrame({
    '부서': ['공연부', '공연부', '공연부', '홍보부', '홍보부', '운영부', '운영부'],
    '항목': ['무대', '음향', '소품', '인쇄', '광고', '식수', '안내'],
    '금액': [1500000, 800000, 200000, 150000, 450000, 180000, 120000],
    '건수': [1, 1, 12, 3, 2, 20, 10]
})

df = club_budget.copy()
fig = px.sunburst(df, path=['부서', '항목'], values='금액', color='부서')
fig.show()

**이 결과에서 알 수 있는 것**

- 공연부의 무대·음향이 큰 지출 항목입니다. 큰 조각 자체가 낭비를 뜻하지는 않습니다.
- 영수증 건수로 바꾸면 작은 금액으로 여러 번 결제한 항목이 두드러집니다.
- 지출 관리와 영수증 처리 업무는 다른 질문이므로 서로 다른 values를 사용합니다.

### 금액 대신 영수증 건수를 넣으면?

In [37]:
# N27
import pandas as pd
import plotly.express as px

# 이 예제에서 사용할 가상 자료를 새로 만듭니다.
club_budget = pd.DataFrame({
    '부서': ['공연부', '공연부', '공연부', '홍보부', '홍보부', '운영부', '운영부'],
    '항목': ['무대', '음향', '소품', '인쇄', '광고', '식수', '안내'],
    '금액': [1500000, 800000, 200000, 150000, 450000, 180000, 120000],
    '건수': [1, 1, 12, 3, 2, 20, 10]
})

df = club_budget.copy()

fig = px.sunburst(df, path=['부서', '항목'], values='건수', color='부서')
fig.show()

**결과를 읽으면**

지출액은 공연부 250만 원이 가장 크지만 영수증은 운영부 30건이 가장 많습니다. 왼쪽은 지출 규모, 오른쪽은 결제 횟수를 비교합니다.

예산을 줄일지는 계획 금액·행사 목적을 추가로 확인한 뒤 판단합니다.

## N17 · 트리맵: 사각형으로 보는 인구 비율

대륙의 사각형 안을 국가별 사각형으로 나누고, 인구가 많은 국가일수록 넓게 표시합니다.

**자료 준비와 그래프 선택**

대륙 안을 국가로 나누고 인구에 비례하는 사각형 면적을 배정합니다. 트리맵은 많은 하위 항목의 비중을 제한된 화면에 담을 때 유용합니다.

**코드에서 확인할 부분**

- path=["continent", "country"]는 대륙 안에 국가를 배치하라는 뜻입니다.
- values="pop"은 사각형 면적을, color="lifeExp"는 색을 정합니다.
- 선버스트와 같은 자료를 사용하지만, 트리맵은 사각형을 채워 비율을 표현합니다.

In [ ]:
# N17
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

df = gap.loc[gap['year'] == 2007].copy()
# 마찬가지로 path에 상위 node가 먼저 와야한다.
fig = px.treemap(df, path=['continent', 'country'],
                 values='pop', color='lifeExp')
fig.show()

**이 결과에서 알 수 있는 것**

- 큰 사각형은 큰 인구입니다. 색은 기대수명이며 면적과 독립된 정보입니다.
- 사각형이 왼쪽에 있다고 서쪽 국가라는 뜻은 아닙니다. 위치는 지리가 아닙니다.
- 같은 자료의 선버스트와 비교해 어느 쪽에서 작은 항목을 찾기 쉬운지 확인합니다.

### 색을 대륙 이름으로 바꾸면?

In [2]:
# N17
import plotly.express as px

# Gapminder에서 2007년 자료만 선택합니다.
gap = px.data.gapminder()
g07 = gap.loc[gap['year'] == 2007].copy()

fig = px.treemap(g07, path=['continent', 'country'],
                 values='pop', color='continent')
fig.show()

**결과를 읽으면**

색이 기대수명 정보를 전달하지 않고 대륙 소속을 강조합니다. 크기에 담긴 인구 정보는 유지됩니다.

정확한 순위와 작은 값 차이가 필요하면 정렬된 막대그래프로 보완합니다.

## N28 · 폴더별 파일: 용량과 개수 비교

강의, 프로젝트, 백업 폴더의 파일 종류별 용량과 개수를 정리한 가상 자료입니다.

**자료 준비와 그래프 선택**

가상 폴더 자료에서 저장 공간을 비교할지, 파일 개수를 비교할지 정합니다. 같은 계층도 면적에 지정한 값이 바뀌면 관리 우선순위가 달라집니다.

**코드에서 확인할 부분**

- path=["폴더", "종류"]는 각 폴더 안에 파일 종류별 사각형을 배치합니다.
- values="용량GB"는 저장 공간을 많이 차지하는 항목을 크게 표시합니다.
- 파일 개수로 바꿔 그리면 어떤 폴더에 파일이 많이 쌓였는지도 비교할 수 있습니다.

### N28 · 폴더별 파일 : 용량과 개수 비교

다음 셀에서 표를 만듭니다. 파일을 불러올 필요는 없습니다. 표를 확인한 뒤 아래 셀에서 그래프를 그립니다.

In [3]:
# N28
import pandas as pd

storage = pd.DataFrame({
    '폴더': ['강의', '강의', '프로젝트', '프로젝트', '백업', '백업'],
    '종류': ['영상', '문서', '데이터', '소스코드', '영상', '문서'],
    '용량GB': [60, 2, 35, 1, 80, 3],
    '파일수': [12, 500, 70, 2000, 16, 750]
})
print(storage)

     폴더    종류  용량GB   파일수
0    강의    영상    60    12
1    강의    문서     2   500
2  프로젝트   데이터    35    70
3  프로젝트  소스코드     1  2000
4    백업    영상    80    16
5    백업    문서     3   750


In [4]:
# N28
import pandas as pd
import plotly.express as px

# 이 예제에서 사용할 가상 자료를 새로 만듭니다.
storage = pd.DataFrame({
    '폴더': ['강의', '강의', '프로젝트', '프로젝트', '백업', '백업'],
    '종류': ['영상', '문서', '데이터', '소스코드', '영상', '문서'],
    '용량GB': [60, 2, 35, 1, 80, 3],
    '파일수': [12, 500, 70, 2000, 16, 750]
})

df = storage.copy()
fig = px.treemap(df, path=['폴더', '종류'], values='용량GB', color='폴더')
fig.show()

**이 결과에서 알 수 있는 것**

- 백업은 83GB로 용량이 가장 크고, 프로젝트는 2,070개로 파일이 가장 많습니다.
- 공간 확보가 목적이면 용량을, 정리할 항목의 양이 궁금하면 개수를 봅니다.
- 큰 폴더라고 바로 삭제하지 않습니다. 중요도와 보존 필요성은 별도 정보입니다.

### 실제 PC에서의 파일 분포

In [ ]:
#실행 하라는거 아님
import random
from pathlib import Path

import pandas as pd
import plotly.express as px

# 1. 조사할 기준 폴더를 지정하세요.
root = Path(r"C:/")

if not root.is_dir():
    raise ValueError("기준 폴더 경로를 확인하세요.")

# 기준 폴더 바로 아래의 폴더 중 최대 5개 선택
folders = [
    p for p in root.iterdir()
    if p.is_dir() and not p.is_symlink()
]
selected = random.sample(folders, min(5, len(folders)))

print("선택된 폴더:")
for folder in selected:
    print(folder.name)

# 2. 확장자로 파일 종류를 분류합니다.
def file_type(path):
    ext = path.suffix.lower()

    if ext in [".mp4", ".avi", ".mov", ".mkv"]:
        return "영상"
    if ext in [".pdf", ".docx", ".pptx", ".xlsx", ".txt", ".hwp"]:
        return "문서"
    if ext in [".csv", ".json", ".parquet", ".npy", ".npz"]:
        return "데이터"
    if ext in [".py", ".ipynb", ".cpp", ".c", ".h", ".js"]:
        return "소스코드"
    if ext in [".png", ".jpg", ".jpeg", ".gif", ".svg"]:
        return "이미지"

    return "기타"

# 3. 선택한 폴더 안의 파일들을 조사합니다.
# 하위 폴더에 들어 있는 파일도 포함합니다.
records = []
skipped = 0

import os

def scan_error(error):
    print("접근하지 못한 경로:", error.filename)

for folder in selected:
    for current, dirs, files in os.walk(
        folder, followlinks=False, onerror=scan_error
    ):
        for name in files:
            path = Path(current) / name

            if path.is_symlink():
                continue

            try:
                size = path.stat().st_size
                records.append({
                    "폴더": folder.name,
                    "종류": file_type(path),
                    "바이트": size
                })
            except OSError:
                skipped += 1

# 4. 폴더·종류별 용량과 파일 수를 합산합니다.
if not records:
    print("집계할 파일이 없습니다.")
else:
    files_df = pd.DataFrame(records)

    storage = (
        files_df.groupby(["폴더", "종류"], as_index=False)
        .agg(
            용량바이트=("바이트", "sum"),
            파일수=("바이트", "size")
        )
    )

    storage["용량GB"] = storage["용량바이트"] / 1_000_000_000
    print(storage[["폴더", "종류", "용량GB", "파일수"]])

    # 면적이 0인 항목은 그래프에서 제외합니다.
    chart_df = storage.loc[storage["용량바이트"] > 0]

    if not chart_df.empty:
        fig = px.treemap(
            chart_df,
            path=["폴더", "종류"],
            values="용량GB",
            color="폴더",
            hover_data=["파일수"],
            title="무작위로 선택한 폴더 5개의 파일 용량"
        )
        fig.show()

print("파일 정보 확인 실패:", skipped, "개")

ValueError: 기준 폴더 경로를 확인하세요.

## N18 · 스트립 플롯: 국가별 기대수명의 분포

대륙별 기대수명이 어느 범위에 모이는지, 국가별 점을 그대로 표시해 비교합니다.

**자료 준비와 그래프 선택**

2007년 기대수명 값을 국가별 점으로 표시하고 대륙을 구분합니다. 국가 수가 많지 않을 때 평균만 요약하지 않고 실제 값의 퍼짐과 예외를 봅니다.

**코드에서 확인할 부분**

- 스트립 플롯(strip plot)은 개별 값을 점으로 표시하는 그래프입니다.
- x="lifeExp"는 기대수명을 가로 위치로, color="continent"는 대륙을 색으로 구분합니다.
- 지터(jitter)는 겹친 점을 알아보기 쉽도록 위아래로 조금 흩뜨리는 표시 방법입니다.

In [ ]:
# N18
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

df = gap.loc[gap['year'] == 2007].copy()
fig = px.strip(df, x='lifeExp', hover_name='country', color='continent')
#jitter로 y축을 약간씩 흔들어줄 수 있음
fig.update_traces(jitter=1.0)
fig.show()

**이 결과에서 알 수 있는 것**

- 점 하나는 국가 하나입니다. 가로 위치는 기대수명, 위아래 흔들림은 겹침을 줄인 표시입니다.
- 대륙 안의 값이 좁게 모이는지, 멀리 떨어진 국가가 있는지 확인합니다.
- 추가 실험처럼 y에 대륙을 지정하면 각 줄이 어느 집단인지 바로 읽을 수 있습니다.

### 색과 함께 대륙 이름을 세로축에도 명시하면?

In [8]:
# N18
import plotly.express as px

# Gapminder에서 2007년 자료만 선택합니다.
gap = px.data.gapminder()
g07 = gap.loc[gap['year'] == 2007].copy()

fig = px.strip(g07, x='lifeExp', y='continent',
               color='continent', hover_name='country')
fig.show()

**결과를 읽으면**

어느 줄이 어느 대륙인지 직접 읽을 수 있습니다. 표본이 적을 때 요약 상자만 보는 것보다 실제 점을 확인하기 좋습니다.

관측이 너무 많아 점이 겹치면 히스토그램으로 구간별 빈도를 비교합니다.

## N19 · 히스토그램: 국가 수와 인구 합계의 차이

기대수명이 비슷한 국가들을 묶은 뒤, 그 국가들에 사는 인구가 얼마나 되는지 합산합니다.

**자료 준비와 그래프 선택**

기대수명을 구간으로 나눈 뒤 국가 수를 셀지, 국가 인구를 더할지 정합니다. 같은 히스토그램이라도 집계가 count인지 sum인지에 따라 세로축의 의미가 달라집니다.

**코드에서 확인할 부분**

- x="lifeExp"는 기대수명을 구간으로 나누는 기준입니다.
- y="pop", histfunc="sum"은 각 구간에 속한 국가들의 인구를 더합니다.
- marginal은 주 그래프의 가장자리에 추가하는 보조 분포 그래프
- marginal="rug"는 각 데이터의 위치를 작은 눈금으로 표시하는 ‘러그 플롯(rug plot)’

In [ ]:
# N19
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

df = gap.loc[gap['year'] == 2007].copy()
fig = px.histogram(df, x='lifeExp', y='pop', histfunc='sum',
    #marginal : 추가 미니 그래프, rug : rug 그래프
    color='continent', marginal='rug', hover_name='country')
fig.show()

## N20 · 밀도 히트맵: 국가들이 모여 있는 구간

GDP와 기대수명을 구간으로 나누고, 각 구간에 국가가 몇 개 있는지 색으로 표시합니다.

**자료 준비와 그래프 선택**

GDP·기대수명을 각각 구간으로 나누고, 구간 조합별 국가 수를 셉니다. 점이 겹치는 산점도에서 관측이 많이 모인 조건을 색으로 요약합니다.

**코드에서 확인할 부분**

- 가로축과 세로축의 구간이 만나는 칸을 셀(cell)이라고 합니다.
- 이 예제에서는 한 칸에 속한 국가가 많을수록 색상 막대에서 큰 값에 해당하는 색이 나타납니다.
- nbinsx와 nbinsy는 원하는 구간 수입니다. 실제 구간 수는 자동 조정될 수 있습니다.

In [ ]:
# N20
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

df = gap.loc[gap['year'] == 2007].copy()
fig = px.density_heatmap(df, x='gdpPercap', y='lifeExp',
                         #x축, y축을 몇개로 나눌 것인지
                         nbinsx=10, nbinsy=10)
fig.show()

**이 결과에서 알 수 있는 것**

- 셀의 색은 그 GDP·기대수명 범위에 속한 국가 수입니다. 확률이나 인구가 아닙니다.
- 구간을 좁히면 세부 위치가 보이지만 빈 칸이 늘어 전체 경향을 놓칠 수 있습니다.
- 10개와 20개 구간을 비교하며, 구간 설정을 바꿔도 많은 국가가 모이는 위치를 찾습니다.

### 구간을 더 잘게 나누면?

In [11]:
# N20
import plotly.express as px

# Gapminder에서 2007년 자료만 선택합니다.
gap = px.data.gapminder()
g07 = gap.loc[gap['year'] == 2007].copy()

fig = px.density_heatmap(g07, x='gdpPercap', y='lifeExp',
                         nbinsx=25, nbinsy=25)
fig.show()

**결과를 읽으면**

세부 위치는 더 보이지만 빈 칸이 많아져 큰 흐름은 약해질 수 있습니다. 142개 국가보다 훨씬 많은 기록에서 집계의 장점이 커집니다.

빈도는 기록 수입니다. 특정 조건의 위험이나 성과를 보려면 다른 집계가 필요합니다.

## N29 · 센서 자료: 측정 횟수와 경보 비율 비교

설비의 온도와 진동을 1,200번 측정하고 경보 여부를 기록한 가상 자료입니다.

**자료 준비와 그래프 선택**

가상 센서의 온도·진동 구간별로 측정 횟수와 경보 비율을 따로 계산합니다. 자주 나타난 조건과 위험 신호가 자주 나온 조건은 같지 않을 수 있습니다.

**코드에서 확인할 부분**

- 가로축은 온도, 세로축은 진동입니다. 색은 그 구간에서 측정한 횟수입니다.
- 경보 열에는 정상 0, 경보 1을 넣었습니다. 이를 구간별로 평균하면 경보 비율을 구할 수 있습니다.
- 자주 측정한 조건과 경보가 자주 발생한 조건을 각각 비교할 수 있습니다.

## N21 · 밀도 등고선: 국가 분포의 윤곽

국가들이 많이 모이는 GDP와 기대수명 범위를 등고선으로 나타내 대륙별 분포를 비교합니다.

**자료 준비와 그래프 선택**

GDP·기대수명의 구간별 국가 수를 구하고 같은 값을 등고선으로 연결합니다. 점 하나보다 집단의 분포 영역과 겹치는 범위가 궁금할 때 사용합니다.

**코드에서 확인할 부분**

- 등고선(contour)은 같은 값을 갖는 위치를 연결한 선입니다. 여기서는 구간별 국가 수를 사용합니다.
- color="continent"는 대륙마다 다른 색의 등고선을 그립니다.
- 구간을 나누고 그 사이를 연결해 그리므로, 윤곽 안의 모든 위치에 실제 국가가 있는 것은 아닙니다.

In [ ]:
# N21
# 등고선이 많다고 표본이 많다는건 아님
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

df = gap.loc[gap['year'] == 2007].copy()
fig = px.density_contour(df, x='gdpPercap', y='lifeExp', color='continent', width=850)
fig.show()

**이 결과에서 알 수 있는 것**

- 등고선은 같은 집계값을 연결한 윤곽입니다. 국가 이동 경로나 개별 국가가 아닙니다.
- 대륙별 윤곽이 겹치는 곳은 두 집단의 수치 범위가 비슷한 곳입니다.
- 오세아니아는 두 국가뿐입니다. 선이 둘러싼 모든 위치에 실제 자료가 있는 것은 아닙니다.

### 집단을 나누지 않고 등고선 내부에 색을 채우면?

In [13]:
# N21
import plotly.express as px

# Gapminder에서 2007년 자료만 선택합니다.
gap = px.data.gapminder()
g07 = gap.loc[gap['year'] == 2007].copy()

fig = px.density_contour(g07, x='gdpPercap', y='lifeExp')
fig.update_traces(contours_coloring='fill', contours_showlabels=True)
fig.show()

**결과를 읽으면**

전체 자료의 집계 수준을 색과 등고선 숫자로 함께 읽습니다. 선의 모양은 구간 설정과 보간에 영향을 받습니다.

관측이 적으면 윤곽보다 원래 산점도를 먼저 확인합니다.

## N22 · 산점도 행렬: 여러 변수의 관계 비교

GDP, 인구, 기대수명을 두 개씩 짝지어 그리면 어떤 변수 사이에 관계가 있는지 한눈에 살펴볼 수 있습니다.

**자료 준비와 그래프 선택**

비교할 수치형 열들을 골라 두 개씩 짝짓습니다. 같은 시점 비교인지 확인합니다. 변수가 많을 때 여러 산점도를 한 번에 탐색하고 유용한 변수 쌍을 고릅니다.

**코드에서 확인할 부분**

- dimensions에는 비교할 수치형 열들을 리스트로 전달합니다.
- 각 칸은 가로축 변수 하나와 세로축 변수 하나를 비교하는 산점도입니다.
- 대각선의 양쪽 칸은 같은 두 변수를 축만 바꿔 그린 것입니다.

In [14]:
# N22
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

fig = px.scatter_matrix(gap,
    dimensions=['lifeExp', 'pop', 'gdpPercap'], color='continent')
fig.show()

**이 결과에서 알 수 있는 것**

- 각 칸은 열 아래 변수와 행 왼쪽 변수의 산점도입니다. 반대쪽 칸은 축만 바뀐 비교입니다.
- 기본 예제는 전체 연도입니다. 한 국가의 여러 시점이 포함되어 있습니다.
- 추가 실험은 2007년만 남깁니다. 국가 간 관계와 시간 변화가 섞인 결과를 구분합니다.

### 2007년에 대한 산점도 행렬

In [15]:
# N22
import plotly.express as px

# Gapminder에서 2007년 자료만 선택합니다.
gap = px.data.gapminder()
g07 = gap.loc[gap['year'] == 2007].copy()

fig = px.scatter_matrix(g07,
    dimensions=['lifeExp', 'pop', 'gdpPercap'], color='continent')
fig.update_traces(diagonal_visible=False, showupperhalf=False)
fig.show()

산점도 행렬은 여러 변수 중 어떤 조합을 자세히 분석할지 고르는 도구입니다

## N30 · 붓꽃 자료: 품종을 구분하는 특징

붓꽃 세 품종의 꽃받침과 꽃잎 크기를 측정한 자료입니다. 어떤 크기 차이로 품종을 구분할 수 있을까요?

**자료 준비와 그래프 선택**

붓꽃 네 가지 측정값을 쌍별로 비교하고, 품종을 색으로 구분합니다. 어떤 측정값 조합이 집단 구분에 유용한지 탐색한 뒤 한 그래프를 골라 설명합니다.

**코드에서 확인할 부분**

- sepal은 꽃받침, petal은 꽃잎입니다. length와 width는 길이와 너비이며 단위는 cm입니다.
- color="species"는 품종별 색을 지정합니다. 점 하나는 꽃 한 송이를 나타냅니다.
- 품종별 점이 잘 나뉘는 변수 쌍을 찾고, 그 두 변수의 산점도를 크게 그려 봅니다.

In [16]:
# N30
import plotly.express as px

df = px.data.iris()
fig = px.scatter_matrix(df,
    dimensions=['sepal_length', 'sepal_width', 'petal_length', 'petal_width'],
    color='species')
fig.update_traces(diagonal_visible=True, showupperhalf=True)
fig.show()

**이 결과에서 알 수 있는 것**

- 꽃잎 길이·너비에서 setosa가 다른 품종과 비교적 잘 나뉩니다.
- 나머지 두 품종은 겹치는 영역도 있습니다. 모든 꽃이 완전히 구분되는 것은 아닙니다.
- 행렬로 변수 쌍을 찾고 선택한 산점도를 크게 그리면 발견한 차이를 전달하기 쉽습니다.

## N23 · 단계구분도: 국가별 기대수명 지도

국가별 기대수명을 지도에 색으로 표시하면, 기대수명이 높은 지역과 낮은 지역을 살펴볼 수 있습니다.

**자료 준비와 그래프 선택**

2007년의 국가 코드와 기대수명을 연결해 국가 경계 안을 색칠합니다. 위치가 중요한 질문, 예를 들어 비슷한 값이 이웃 지역에 모이는지 살펴봅니다.

**코드에서 확인할 부분**

- 단계구분도(choropleth map)는 각 지역 안을 통계값에 따라 다른 색으로 채운 지도입니다.
- locations="iso_alpha"는 국가 코드를 읽습니다. ISO-3은 KOR, USA 같은 세 글자 코드입니다.
- color="lifeExp"는 기대수명으로 색을 정하고, range_color는 색에 대응하는 값의 범위를 정합니다.

In [132]:
# N23
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

df = gap.loc[gap['year'] == 2007].copy()
fig = px.choropleth(df, locations='iso_alpha',
    locationmode='ISO-3', color='lifeExp', hover_name='country',
    range_color=[20, 85], color_continuous_scale='Viridis')

fig.show()

### 연도별 변화를 보여주기 위한 애니메이션

In [133]:
# N23
import plotly.express as px

# 국가별 인구·기대수명·GDP 자료를 불러옵니다.
gap = px.data.gapminder()

fig = px.choropleth(gap, locations='iso_alpha', color='lifeExp',
    hover_name='country', animation_frame='year',
    range_color=[20, 85], color_continuous_scale='Viridis')
fig.show()

## N31 · 지역별 건수: 인구를 고려한 비교

미국, 일본, 한국의 가상 건수를 비교합니다. 인구가 다른 지역을 같은 기준으로 비교하려면 어떻게 해야 할까요?

**자료 준비와 그래프 선택**

가상 지역 자료에서 총건수와 인구 10만 명당 건수를 각각 계산합니다. 처리할 총규모와 인구 차이를 고려한 발생 정도는 서로 다른 비교입니다.

**코드에서 확인할 부분**

- 첫 지도는 전체 건수를 색으로 표시합니다. 실제 통계가 아닌 설명용 가상 자료입니다.
- 인구 10만 명당 건수는 전체 건수를 인구로 나눈 뒤 100,000을 곱해 계산합니다.
- 전체 건수는 총규모를, 인구당 건수는 인구 차이를 고려한 발생 정도를 보여 줍니다.

### N31 · 직접 만든 데이터를 지도에 적용


In [17]:
# N31
import pandas as pd

regional_demo = pd.DataFrame({
    '지역': ['한국', '일본', '미국'],
    'iso': ['KOR', 'JPN', 'USA'],
    '인구': [50000000, 125000000, 330000000],
    '건수': [5000, 7500, 9900]
})
print(regional_demo)

   지역  iso         인구    건수
0  한국  KOR   50000000  5000
1  일본  JPN  125000000  7500
2  미국  USA  330000000  9900


In [18]:
# N31
import pandas as pd
import plotly.express as px

# 이 예제에서 사용할 가상 자료를 새로 만듭니다.
regional_demo = pd.DataFrame({
    '지역': ['한국', '일본', '미국'],
    'iso': ['KOR', 'JPN', 'USA'],
    '인구': [50000000, 125000000, 330000000],
    '건수': [5000, 7500, 9900]
})  # 인구와 건수 모두 비교 방법을 설명하기 위한 가상 값입니다.

df = regional_demo.copy()
fig = px.choropleth(df, locations='iso', color='건수', hover_name='지역',
                   color_continuous_scale='Blues')
fig.show()

### 인구 10만 명당 건수로 바꾸면?

In [19]:
# N31
import pandas as pd
import plotly.express as px

# 이 예제에서 사용할 가상 자료를 새로 만듭니다.
regional_demo = pd.DataFrame({
    '지역': ['한국', '일본', '미국'],
    'iso': ['KOR', 'JPN', 'USA'],
    '인구': [50000000, 125000000, 330000000],
    '건수': [5000, 7500, 9900]
})  # 인구와 건수 모두 비교 방법을 설명하기 위한 가상 값입니다.

df = regional_demo.copy()

df['10만명당'] = df['건수'] / df['인구'] * 100000
fig = px.choropleth(df, locations='iso', color='10만명당',
                   hover_name='지역', color_continuous_scale='Blues')
fig.show()
print(df[['지역', '건수', '10만명당']])

   지역    건수  10만명당
0  한국  5000   10.0
1  일본  7500    6.0
2  미국  9900    3.0


## N24 · 평행 범주: 식사 기록의 여러 특성 비교

식당의 식사 기록으로 요일, 점심·저녁, 일행 수 등이 어떤 조합으로 나타나는지 살펴봅니다.

**자료 준비와 그래프 선택**

식사 기록의 범주형 특성을 선택하고, 조합별 기록 수를 띠로 연결합니다. 여러 분류가 함께 나타나는 조합을 볼 때 사용합니다. 전체 변수 모두를 넣을 필요는 없습니다.

**코드에서 확인할 부분**

- 평행 범주 다이어그램은 여러 범주형 변수를 나란히 놓고 같은 기록의 항목들을 띠로 연결합니다.
- dimensions는 왼쪽부터 표시할 변수들의 순서입니다. 같은 조합이 많을수록 띠가 굵어집니다.
- 이 자료의 size는 식사 일행 수입니다. color="size"는 일행 수를 띠의 색으로 표시합니다.

In [20]:
# N24
import plotly.express as px

df = px.data.tips()

fig = px.parallel_categories(df,
    dimensions=['sex', 'smoker', 'day', 'time', 'size'],
    color='size', color_continuous_scale=px.colors.sequential.Inferno)
fig.show()

**이 결과에서 알 수 있는 것**

- Sat·Dinner·2를 잇는 띠는 토요일 저녁 두 명의 식사 기록에 해당합니다.
- 축을 요일·식사 시간·일행 수로 줄이면 조합을 따라가기 쉽습니다.

### 축을 요일·식사 시간·인원 세 개로 줄이면?

In [21]:
# N24
import plotly.express as px

df = px.data.tips()

fig = px.parallel_categories(df, dimensions=['day', 'time', 'size'],
    color='size', color_continuous_scale='Inferno')
fig.show()

## N25 · 지도 산점도: 차량 공유 서비스 현황

몬트리올의 차량 공유 서비스 자료로 지역별 이용 가능량과 자료에 기록된 피크 시간을 살펴봅니다.

**자료 준비와 그래프 선택**

지역 중심의 위도·경도에 점을 놓고 car_hours와 peak_hour를 표시합니다. 위치와 함께 규모·시간대를 보고, 어떤 지역을 더 조사할지 찾습니다.

**코드에서 확인할 부분**

- lat와 lon은 위도와 경도입니다. 이 예제는 지역을 대표하는 중심 좌표에 점을 찍습니다.
- car_hours는 이용 가능량, peak_hour는 자료에 기록된 피크 시간입니다. 대여 횟수는 아닙니다.
- size와 color로 두 값을 구분하고, zoom과 center로 지도의 확대 수준과 중심을 정합니다.

In [22]:
# N25
import plotly.express as px

df = px.data.carshare()
print(df.head())
fig = px.scatter_map(df, lat='centroid_lat', lon='centroid_lon',
    color='peak_hour', size='car_hours', size_max=15, zoom=10,
    center={'lat': 45.5234, 'lon': -73.5918},
    color_continuous_scale=px.colors.cyclical.IceFire,
    map_style='carto-positron')
fig.show()

   centroid_lat  centroid_lon    car_hours  peak_hour
0     45.471549    -73.588684  1772.750000          2
1     45.543865    -73.562456   986.333333         23
2     45.487640    -73.642767   354.750000         20
3     45.522870    -73.595677   560.166667         23
4     45.453971    -73.738946  2836.666667         19


### 점 크기 대신 색으로 이용 가능량만 강조한다면?

In [23]:
# N25
import plotly.express as px

df = px.data.carshare()

fig = px.scatter_map(df, lat='centroid_lat', lon='centroid_lon',
    color='car_hours', size='car_hours', zoom=10,
    center={'lat': 45.5234, 'lon': -73.5918},
    map_style='carto-positron', color_continuous_scale='Viridis')
fig.show()

## N26 · 극좌표 산점도: 바람의 방향과 빈도

바람이 어느 방향에서 얼마나 자주 불었는지, 풍향과 바람 세기별 빈도를 원형 그래프로 살펴봅니다.

**자료 준비와 그래프 선택**

풍향을 각도에, 해당 풍향·세기의 빈도를 중심에서의 거리에 놓습니다. 북쪽과 북동쪽처럼 방향의 이웃 관계가 중요할 때 원형 배치가 자연스럽습니다.

**코드에서 확인할 부분**

- 극좌표는 각도 θ와 중심에서의 거리 r로 점의 위치를 나타내는 좌표계입니다.
- theta="direction"은 풍향, r="frequency"는 그 방향과 세기에 해당하는 빈도입니다.
- 바람 세기는 color와 symbol로 구분합니다. Plasma_r의 _r은 색 순서를 뒤집는 표시입니다.

In [24]:
# N26
import plotly.express as px

df = px.data.wind()
print(df.head())
fig = px.scatter_polar(df, r='frequency', theta='direction',
    color='strength', symbol='strength',
    color_discrete_sequence=px.colors.sequential.Plasma_r)
fig.show()

  direction strength  frequency
0         N      0-1        0.5
1       NNE      0-1        0.6
2        NE      0-1        0.5
3       ENE      0-1        0.4
4         E      0-1        0.4


### 점 대신 방향별 누적 막대로 보면?

In [25]:
# N26
import plotly.express as px

df = px.data.wind()

fig = px.bar_polar(df, r='frequency', theta='direction', color='strength',
    color_discrete_sequence=px.colors.sequential.Plasma_r)
fig.show()

**결과를 읽으면**

각 방향의 총빈도와 세기 구간별 기여를 함께 볼 수 있습니다. 반지름 길이로 값을 읽고 부채꼴 면적 자체를 빈도로 읽지 않습니다.

일반 항목의 크기만 비교하려면 원형 배치보다 막대그래프가 더 읽기 쉽습니다.